In [1]:
import signal
import grpc
import numpy as np
from time import sleep
from concurrent import futures

import grpc
from interfaces.rtde_socket_client import RTDESocketClient, rtde_data
from interfaces.control_socket_client import ControlSocketClient, control_data, common_data
from interfaces.impl import common_msgs_pb2 as common_data
from interfaces.impl import hri_msgs_pb2 as conty_data
from interfaces.impl import hri_pb2_grpc as conty_grpc
from interfaces.impl import shared_msgs_pb2 as shared_data

ip = '192.168.0.147'

control = ControlSocketClient(ip, port=20001)
rtde = RTDESocketClient(ip, port=20004)


[INFO] 2024-01-29 20:32:05,222, Logger Started


In [40]:
conty_channel = grpc.insecure_channel("{}:{}".format(ip, 20131))
conty_stub = conty_grpc.HRIStub(conty_channel)

In [24]:
conty_stub.ContyInit(conty_data.ContyInitReq())

robot_name: "NRMK-Indy7"
robot_dof: 6
robot_sn: "Robot-151"
robot_payload: 7.0
cb_ip: "192.168.0.147"
cb_sn: "CB-133"
controller_ver: "3.2.0"
controller_date: "2024.01.29"
server_ver: "3.2.0"
server_date: "2024.01.29"
config_path: "/home/user/dev/runtest/Release/IndyDeployment/../IndyConfigurations/Cobot/Params/"
program_path: "/home/user/dev/runtest/Release/IndyDeployment/ProgramScripts"
index_program_path: "/home/user/dev/runtest/Release/IndyDeployment/ProgramScripts/index"
server_log_path: "/home/user/dev/runtest/Release/IndyDeployment/LogData/Server/"
rt_log_path: "/home/user/dev/runtest/Release/IndyDeployment/LogData/"
io_fw_ver: "-1.0B"
core_fw_vers: "-1.0C"
core_fw_vers: "-1.0C"
core_fw_vers: "-1.0C"
core_fw_vers: "-1.0C"
core_fw_vers: "-1.0C"
core_fw_vers: "-1.0C"
endtool_fw_ver: "-1.0E"
joint_limits: 175.0
joint_limits: 175.0
joint_limits: 175.0
joint_limits: 175.0
joint_limits: 175.0
joint_limits: 215.0

In [42]:
vision_servers  = conty_stub.GetVisionServerList(conty_data.GetVisionObjectListReq()).vision_servers 
vision_servers 

[name: "eye"
ip: "192.168.0.114"
port: 10511
]

In [43]:
objects = conty_stub.GetVisionObjectList(
    conty_data.GetVisionObjectListReq(vision_server=vision_servers[0])).objects
objects

['marker']

In [45]:
conty_stub.GetVisionDetection(
    conty_data.VisionRequest(
        vision_server=vision_servers[0],
        object = objects[0],
        frame_type=shared_data.OBJECT
    ))

frame: -10.648191452026367
frame: -158.97686767578125
frame: -33.300498962402344
frame: -0.3825732469558716
frame: 2.498974084854126
frame: 162.19630432128906
object: "marker"
detected: true
passed: true

In [46]:
conty_stub.GetVisionDetection(
    conty_data.VisionRequest(
        vision_server=vision_servers[0],
        object = objects[0],
        frame_type=shared_data.END_EFFECTOR
    ))

frame: 63.76224136352539
frame: -184.9188690185547
frame: 126.364501953125
frame: -177.96604919433594
frame: -3.0736804008483887
frame: -18.67030906677246
frame_type: END_EFFECTOR
object: "marker"
detected: true
passed: true

In [ ]:
task_pos = Common.Utils.pos_to_transform(framework_data['control_data']['p'])  # Trt (mm)
ref_frame = Common.Utils.pos_to_transform(framework_data['control_data']['ref_frame'])  # Tbr (mm)
tool_frame = Common.Utils.pos_to_transform(framework_data['control_data']['tool_frame'])  # Tet (mm)

In [11]:

from math import *

def rot_axis(axis, degree):
    th = radians(degree)
    if axis == 1:
        rot_matrix = np.asarray([[1, 0, 0], [0, cos(th), -sin(th)], [0, sin(th), cos(th)]])
    elif axis == 2:
        rot_matrix = np.asarray([[cos(th), 0, sin(th)], [0, 1, 0], [-sin(th), 0, cos(th)]])
    elif axis == 3:
        rot_matrix = np.asarray([[cos(th), -sin(th), 0], [sin(th), cos(th), 0], [0, 0, 1]])
    else:
        rot_matrix = np.identity
    return rot_matrix


def euler_to_rotm(uvw):
    rx = uvw[0]
    ry = uvw[1]
    rz = uvw[2]
    return np.matmul(np.matmul(rot_axis(3, rz), rot_axis(2, ry)), rot_axis(1, rx))

def pos_to_transform(p):
    xyz = p[:3]
    uvw = p[3:]
    transform_matrix = np.identity(4)
    transform_matrix[:3, :3] = euler_to_rotm(uvw)
    transform_matrix[:3, 3] = xyz[:]
    return transform_matrix

In [22]:
framework_data = rtde.GetControlData()
task_pos = framework_data['p']
ref_frame = framework_data['ref_frame']
tool_frame = framework_data['tool_frame']

In [23]:

task_pos = pos_to_transform(task_pos)  # Trt (mm)
ref_frame = pos_to_transform(ref_frame)  # Tbr (mm)
tool_frame = pos_to_transform(tool_frame)  # Tet (mm)

In [21]:

# task_pos_bak = task_pos
# ref_frame_bak = ref_frame
# tool_frame_bak = tool_frame

In [24]:
task_pos

array([[-9.99999724e-01,  4.17197163e-04,  6.15096334e-04,
        -5.76580700e+02],
       [ 4.16784534e-04,  9.99999688e-01, -6.70811869e-04,
         5.38331000e-02],
       [-6.15376003e-04, -6.70555321e-04, -9.99999586e-01,
        -1.33343250e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [ ]:
# task_pos = Trb'*Tre*Tet
#

In [35]:
np.round(task_pos - np.matmul(np.matmul(np.linalg.inv(ref_frame), task_pos_bak), tool_frame), 4)

array([[ 0.    ,  0.    , -0.    , -0.0003],
       [ 0.    , -0.    , -0.    , -0.0002],
       [ 0.    , -0.    , -0.    , -0.0002],
       [ 0.    ,  0.    ,  0.    ,  0.    ]])

In [36]:
robot_pos = np.matmul(np.matmul(ref_frame, task_pos), np.linalg.inv(tool_frame))

In [37]:
robot_pos

array([[-9.99999966e-01,  2.42678630e-04,  9.14928275e-05,
        -2.26714180e+02],
       [ 2.42681132e-04,  9.99999970e-01,  2.73356180e-05,
        -1.86457821e+02],
       [-9.14861910e-05,  2.73578207e-05, -9.99999995e-01,
         5.38998735e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [38]:
task_pos_bak

array([[-9.99999967e-01,  2.41551055e-04,  9.16497362e-05,
        -2.26713880e+02],
       [ 2.41553565e-04,  9.99999970e-01,  2.73795317e-05,
        -1.86457640e+02],
       [-9.16431199e-05,  2.74016691e-05, -9.99999995e-01,
         5.38998900e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [26]:
np.matmul(np.matmul(ref_frame_bak, task_pos_bak), 

array([[-9.99999967e-01,  2.41551055e-04,  9.16497362e-05,
        -2.26713880e+02],
       [ 2.41553565e-04,  9.99999970e-01,  2.73795317e-05,
        -1.86457640e+02],
       [-9.16431199e-05,  2.74016691e-05, -9.99999995e-01,
         5.38998900e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [17]:
np.matmul(np.matmul(np.linalg.inv(ref_frame), task_pos), tool_frame)

array([[-9.99999967e-01,  2.41551055e-04,  9.16497362e-05,
        -2.26713880e+02],
       [ 2.41553565e-04,  9.99999970e-01,  2.73795317e-05,
        -1.86457640e+02],
       [-9.16431199e-05,  2.74016691e-05, -9.99999995e-01,
         5.38998900e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [13]:
task_pos

[-226.71388, -186.45764, 538.9989, 179.99843, 0.005250764, 179.98616]

In [9]:
conty_stub.GetJointControlGain(conty_data.GetJointControlGainReq())

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNKNOWN
	details = "Exception calling application: 0"
	debug_error_string = "{"created":"@1706517358.773000000","description":"Error received from peer ipv4:192.168.214.20:20131","file":"src/core/lib/surface/call.cc","file_line":1070,"grpc_message":"Exception calling application: 0","grpc_status":2}"
>

## Device protocol

In [3]:
conty_stub.GetTeleOpDevice(conty_data.Empty())

name: "tele2"
type: VIVE
ip: "192.168.8.184"
port: 20500

In [4]:
conty_stub.GetTeleOpState(conty_data.Empty())

In [5]:
conty_stub.ConnectTeleOpDevice(conty_data.TeleOpDevice(
    name="tele", type=control_data.TeleOpDevice.VIVE,
    ip="192.168.8.184", port=20500
    )
)

In [6]:
conty_stub.ReadTeleOpInput(conty_data.Empty()) 

tpos: -358.3995361328125
tpos: -196.8184814453125
tpos: 525.206298828125
tpos: -178.9840850830078
tpos: -0.9875966906547546
tpos: 179.99417114257812

In [7]:
conty_stub.DisConnectTeleOpDevice(conty_data.Empty())

In [8]:
conty_stub.GetTeleFileList(conty_data.Empty())

files: "ttt"
files: "tele"
files: "vvv"

In [9]:
conty_stub.SaveTeleMotion(conty_data.TeleFileReq(name="tele"))

In [10]:
conty_stub.LoadTeleMotion(conty_data.TeleFileReq(name="tele"))

### Test Joint Jog

In [23]:
conty_stub.StartTeleJogJ(conty_data.Empty())
vals = np.zeros(6)

In [24]:
conty_stub.GetTeleOpState(conty_data.Empty())

mode: TELE_RAW
method: TELE_JOINT_RELATIVE

In [25]:
conty_stub.MoveTeleJ(
    conty_data.MoveTeleJReq(jpos=[-30,0,0,0,0,0], 
                            vel_ratio=0.1, acc_ratio=0.1))

In [26]:
conty_stub.StopTeleOp(conty_data.Empty())

msg: "TeleOp Disabled"

### Test Task Jog

In [85]:
conty_stub.StartTeleJogL(conty_data.Empty())

msg: "TeleOp Enabled"

In [86]:
conty_stub.MoveTeleL(
    conty_data.MoveTeleLReq(tpos=[100,0,0,0,0,0], 
                            vel_ratio=0.1, acc_ratio=0.1))

In [29]:
conty_stub.StopTeleOp(conty_data.Empty())

msg: "TeleOp Disabled"

### Test Calibration

In [11]:
import matplotlib.pyplot as plt

In [11]:
conty_stub.ConnectTeleOpDevice(conty_data.TeleOpDevice(
    name="tele", type=control_data.TeleOpDevice.VIVE,
    ip="192.168.8.184", port=20500
    )
)

In [21]:
conty_stub.ReadTeleOpInput(conty_data.Empty()) 

tpos: -358.3992614746094
tpos: -196.81834411621094
tpos: 525.2061157226562
tpos: -178.984130859375
tpos: -0.987571120262146
tpos: 179.99417114257812

In [22]:
conty_stub.StartTeleCalib(conty_data.Empty())

msg: "TeleOp Enabled"

In [23]:
conty_stub.DisConnectTeleOpDevice(conty_data.Empty())

In [24]:
control.StopTeleOp()

{'msg': 'TeleOp Disabled', 'code': '0'}

### Test Record

In [26]:
conty_stub.StartTeleRecord(conty_data.Empty())

msg: "TeleOp Enabled"

In [5]:
control_dat = rtde.GetControlData()
jpos = list(control_dat['q'])
tpos = list(control_dat['p'])
vals = tpos
vals

[-355.26505, -197.67221, 522.798, -178.9875, -0.9946007, 179.99483]

In [6]:
control.MoveTeleL(vals)

{'code': '0', 'msg': ''}

In [7]:
for _ in range(100):
    sleep(0.01)
    vals[0] -= 1
    control.MoveTeleL(vals)

In [8]:
for _ in range(100):
    sleep(0.01)
    vals[0] += 1
    control.MoveTeleL(vals)

In [9]:
for _ in range(100):
    sleep(0.01)
    vals[1] -= 1
    control.MoveTeleL(vals)

In [10]:
for _ in range(100):
    sleep(0.01)
    vals[1] += 1
    control.MoveTeleL(vals)

In [11]:
for _ in range(100):
    sleep(0.01)
    vals[2] -= 1
    control.MoveTeleL(vals)

In [12]:
for _ in range(100):
    sleep(0.01)
    vals[2] += 1
    control.MoveTeleL(vals)

In [13]:
for _ in range(100):
    sleep(0.01)
    vals[3] -= 0.3
    control.MoveTeleL(vals)

In [14]:
for _ in range(100):
    sleep(0.01)
    vals[3] += 0.3
    control.MoveTeleL(vals)

In [15]:
for _ in range(100):
    sleep(0.01)
    vals[4] -= 0.3
    control.MoveTeleL(vals)

In [16]:
for _ in range(100):
    sleep(0.01)
    vals[4] += 0.3
    control.MoveTeleL(vals)

In [17]:
for _ in range(100):
    sleep(0.01)
    vals[5] -= 0.3
    control.MoveTeleL(vals)

In [18]:
for _ in range(100):
    sleep(0.01)
    vals[5] += 0.3
    control.MoveTeleL(vals)

In [41]:
conty_stub.StopTeleOp(conty_data.Empty())

msg: "TeleOp Disabled"

### Test Play

In [9]:
conty_stub.StartTelePlay(conty_data.Empty())

msg: "TeleOp Enabled"

In [43]:
conty_stub.StopTeleOp(conty_data.Empty())

msg: "TeleOp Disabled"

### Test Save & Load

In [13]:
# conty_stub.SaveTeleMotion(conty_data.TeleFileReq(name="tele"))

In [8]:
conty_stub.LoadTeleMotion(conty_data.TeleFileReq(name="tele"))

In [12]:
conty_stub.PlayProgram(conty_data.PlayProgramReq(file_dir='/home/user/dev/yDeployment/ProgramScripts/tele_program.indy7.json'))

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = ""
	debug_error_string = "{"created":"@1703129071.446000000","description":"Error received from peer ipv4:192.168.0.105:20131","file":"src/core/lib/surface/call.cc","file_line":1070,"grpc_message":"","grpc_status":14}"
>